# AgriAI Advisor - Crop Disease Detection Model Training
This notebook demonstrates fine-tuning a **MobileNetV2** transfer learning model on the **PlantVillage** dataset for crop leaf disease classification.

In [ ]:
import os
import json
import numpy as np
import tensorflow as tf
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout, Input
from tensorflow.keras.models import Model

IMG_SIZE = (224, 224)
BATCH_SIZE = 32
DATASET_DIR = '../dataset/PlantVillage/'
MODEL_SAVE_PATH = '../model/disease_model.keras'
CLASS_NAMES_SAVE_PATH = '../model/class_names.json'

In [ ]:
# 1. Data Generators & Augmentation
train_datagen = ImageDataGenerator(
    rescale=1./127.5, samplewise_center=True,
    rotation_range=20, width_shift_range=0.2,
    height_shift_range=0.2, shear_range=0.15,
    zoom_range=0.15, horizontal_flip=True, validation_split=0.2
)

train_generator = train_datagen.flow_from_directory(
    DATASET_DIR, target_size=IMG_SIZE,
    batch_size=BATCH_SIZE, class_mode='categorical', subset='training'
)

val_generator = train_datagen.flow_from_directory(
    DATASET_DIR, target_size=IMG_SIZE,
    batch_size=BATCH_SIZE, class_mode='categorical', subset='validation'
)

In [ ]:
# 2. Save class mapping dictionary
class_indices = train_generator.class_indices
idx_to_class = {v: k for k, v in class_indices.items()}
with open(CLASS_NAMES_SAVE_PATH, 'w') as f:
    json.dump(idx_to_class, f, indent=2)
print('Saved class names mapping.')

In [ ]:
# 3. Build Transfer Learning Model with MobileNetV2
inputs = Input(shape=(224, 224, 3))
base_model = MobileNetV2(input_tensor=inputs, weights='imagenet', include_top=False)
base_model.trainable = False

x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dropout(0.2)(x)
outputs = Dense(len(class_indices), activation='softmax')(x)

model = Model(inputs=inputs, outputs=outputs)
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
model.summary()

In [ ]:
# 4. Train model
history = model.fit(
    train_generator, validation_data=val_generator,
    epochs=10
)

# 5. Save Model Keras Artifact
model.save(MODEL_SAVE_PATH)
print(f'Model saved to {MODEL_SAVE_PATH}')